In [ ]:
from dataclasses import dataclass
from src import *
from src.internals import *
from src.shading import ExtendedBlinnPhongShader
import math

# Integrators
---
### Simple ray caster

In [ ]:
@dataclass
class RayCaster(Integrator):
    """
    A simple ray caster that only computes local shading at the first hit point, without any recursion.
    """
    scene: Scene
    shader: LocalShading | None = None

    def __post_init__(self):
        if self.shader is None:
            self.shader = ExtendedBlinnPhongShader()

    def cast_ray(self, ray: Ray, depth: int | None = None) -> Color:
        hit = self.scene.intersect(ray)

        if hit is not None:
            return self.shader.shade_multiple_lights(
                hit,
                lights=self.scene.lights,
                view_dir=-ray.direction,   # ray to camera is opposite of ray direction
                scene=self.scene
            )

        return Color.background_color(ray, self.scene.skybox)

In [ ]:
point_light = PointLight(position=Vertex(5, 5, 0), intensity=1)
lights = [point_light, AmbientLight(intensity=0.1)]

objects = [
    Object(
        geometry=Sphere(),
        material=PhongMaterial(base_color=Color(1, 0, 0)),
    ).translate(0, 0, -5),
    Object(
        geometry=Plane(),
        material=PhongMaterial(base_color=Color(0.5, 0.5, 0.5)),
    ).translate(0, -1, 0).rotate_x(math.pi / 2),
]

scene = Scene(
    objects=objects,
    lights=lights,
    camera=PinholeCamera(
        origin=Vertex(0, 0, 0),
        direction=Vector(0, 0, -1),
        fov_deg=50,
    )
)
